# Estimador de Doble Lasso

Estimaremos la brecha de genero en la predicción de salarios. Para ello estimamos la siguiente regresión

\begin{equation}
lw=\alpha M+\beta'W+\epsilon
\end{equation}

donde $M$ es una variable indicadora de mujer y $W$ contiene otros determinantes del salario

## Datos

Usaremos los mismos datos del notebook de predicciónd de salarios (sesión 01). 

In [ ]:
suppressWarnings(
pacman::p_load(xtable, dplyr,hdm,sandwich,lmtest)
)

In [ ]:
file<-"https://raw.githubusercontent.com/andvarga-eco/econometrics_ml/refs/heads/main/data/geih_24_baq.csv"
data<-read.csv(file)

In [ ]:
data<-data|>filter(P6040 %in% 25:64)|>filter(P3042!=99)
data<-na.omit(data)
data<-data|>filter(P6800<=60)|>mutate(w=INGLABO/(P6800*4),lw=log(w),
                                      edad2=P6040^2, edad3=P6040^3, edad4=P6040^4,
                                    M=ifelse(P3271==2,1,0))

## Partialling-out

Antes de realizar el procedimiento de doble Lasso, realizaremos el partialling-out para verificar que obtenemos el mismo parámetro que si lo estimamos directamente por MCO

In [ ]:
basic<-lw~M+P6040+C(P3042)+C(RAMA2D_R4)
YW<-lw~P6040+C(P3042)+C(RAMA2D_R4)
DW<-M~P6040+C(P3042)+C(RAMA2D_R4)

Estimación directa por MCO

In [ ]:
options(warn = -1)
basicreg<-lm(basic,data)
reg_basic<-coeftest(basicreg, vcov = vcovHC(basicreg, type = "HC1"))
cat("Coef. M:",round(reg_basic["M","Estimate"],4),
" | EE:", round(reg_basic["M", "Std. Error"], 4), "\n")

In [ ]:
Haciendo partialling-out

In [ ]:
YWreg<-lm(YW,data)
Y_residual<-residuals(YWreg)
DWreg<-lm(DW,data)
D_residual<-residuals(DWreg)
p_out<-lm(Y_residual~D_residual)
res_p_out<-coeftest(p_out, vcov = vcovHC(p_out, type = "HC1"))
cat("Coef. M:",round(res_p_out["D_residual","Estimate"],4),
" | EE:", round(res_p_out["D_residual", "Std. Error"], 4), "\n")

Verificamos que el procedimiento de partialling-out recupera el coeficiente de interés. Ahora, usaremos la misma idea con el estimado de Lasso

## Doble Lasso con partialling-out

A diferencia del notebook anterior, lasso_wage_predict.ipynb, en este no haces sample-splitting. Por lo tanto, usamos el parámetro de penalidad, $\lambda$ de acuerdo al criterio teórico, $\lambda=2c\hat\sigma\sqrt{n}z_{(1-a)/2p}$

In [ ]:
y <- data$lw
W <- model.matrix(~ P6040 + factor(P3042) + factor(RAMA2D_R4), data = data)[, -1]  # [, -1] quita el intercepto
D <- data$M

In [ ]:
YW_lasso<-rlasso(W,y,post=FALSE)
yhat_lasso<-predict(YW_lasso,as.data.frame(W))
yres<-y-as.numeric(yhat_lasso)

DW_lasso<-rlasso(W,D,post=FALSE)
Dhat_lasso<-predict(DW_lasso,as.data.frame(W))
Dres<-D-as.numeric(Dhat_lasso)


In [ ]:
p_out_lasso<-lm(yres~Dres)
res_p_out_lasso<-coeftest(p_out_lasso, vcov = vcovHC(p_out_lasso, type = "HC1"))
cat("Coef. M:",round(res_p_out_lasso["Dres","Estimate"],4),
" | EE:", round(res_p_out_lasso["Dres", "Std. Error"], 4), "\n")

### Repliquemos el ejercicio con la especificación flexible

In [ ]:
flex<-lw~M+C(P3042)+C(RAMA2D_R4)+(P6040+edad2+edad3+edad4)*(C(P3042)+C(RAMA2D_R4))

In [ ]:
options(warn = -1)
flexreg<-lm(flex,data)
reg_flex<-coeftest(flexreg, vcov = vcovHC(flexreg, type = "HC1"))
cat("Coef. M:",round(reg_flex["M","Estimate"],4),
" | EE:", round(reg_flex["M", "Std. Error"], 4), "\n")

In [ ]:
cat("Number of regressors in the flex model:", length(flexreg$coef), "\n")

In [ ]:
W_flex<-model.matrix(~C(P3042)+C(RAMA2D_R4)+(P6040+edad2+edad3+edad4)*(C(P3042)+C(RAMA2D_R4)), data = data)[, -1]

In [ ]:
YW_lasso<-rlasso(W_flex,y,post=FALSE)
yhat_lasso<-predict(YW_lasso,as.data.frame(W_flex))
yres<-y-as.numeric(yhat_lasso)

DW_lasso<-rlasso(W_flex,D,post=FALSE)
Dhat_lasso<-predict(DW_lasso,as.data.frame(W_flex))
Dres<-D-as.numeric(Dhat_lasso)

p_out_lasso<-lm(yres~Dres)
res_p_out_lasso<-coeftest(p_out_lasso, vcov = vcovHC(p_out_lasso, type = "HC1"))
cat("Coef. M:",round(res_p_out_lasso["Dres","Estimate"],4),
" | EE:", round(res_p_out_lasso["Dres", "Std. Error"], 4), "\n")

Note que los resultados son similares entre Doble Lasso y MCO, esto en coherenca con la situación particular de este ejercicio, donde $p/n$ es pequeño

## Ejercicio

1. Replique el análisis de múltiples coeficientes, *Discovering heterogeneity in the wage gap analysis*, usando los datos para Barranquilla de este notebook
2. Replique el análisis de hipótesis de convergencia presentado en el cap. 4 del libro Causal ML
3. Verifique doble Lasso con partialling-out es equivalente a la doble selección usando Lasso. Para ello realice el procedimiento *double selection* de la sección 4.5 del libro Causal ML